### Joins & Unions in Apache Spark DataFrame/Dataset

#### Summary

![image_1775383760327.png](./image_1775383760327.png "image_1775383760327.png")

![image_1775383784963.png](./image_1775383784963.png "image_1775383784963.png")

Sample Data Setup

In [0]:

employees = [
    (1, "Alice",   "IN", 10),
    (2, "Bob",     "US", 20),
    (3, "Charlie", "IN", 10),
    (4, "Diana",   "EU", 30),
    (5, "Eve",     "US", 20),
    (6, "Frank",   "IN", 99),   # dept 99 doesn't exist
]
emp_df = spark.createDataFrame(employees, ["emp_id", "name", "country", "dept_id"])

departments = [
    (10, "Engineering",  80000),
    (20, "Marketing",    60000),
    (30, "Finance",      70000),
    (40, "HR",           50000),   # no employees in HR
]
dept_df = spark.createDataFrame(departments, ["dept_id", "dept_name", "budget"])

#### 1. INNER JOIN — Only Matching Rows####

In [0]:
emp_df.join(dept_df, "dept_id", "inner").show()

> Frank (dept 99) dropped — no match. HR (dept 40) dropped — no employees.

#### 2. LEFT JOIN — All Left Rows + Matching Right

In [0]:
emp_df.join(dept_df, "dept_id", "left").show()

> All employees kept. Frank gets null for dept columns.

#### 3. RIGHT JOIN — All Right Rows + Matching Left

In [0]:
emp_df.join(dept_df, "dept_id", "right").show()

> All departments kept. HR gets null for employee columns.

#### 4. FULL OUTER JOIN — All Rows from Both Sides

In [0]:
emp_df.join(dept_df, "dept_id","full" ).show()

> No rows dropped from either side.

#### 5. LEFT SEMI JOIN — Left Rows That Have a Match (no right columns)

In [0]:
emp_df.join(dept_df, "dept_id", "left_semi").show()

> Like INNER JOIN but **only returns left columns**. Frank excluded (no match). Think of it as a filter using the right table.

#### 6. LEFT ANTI JOIN — Left Rows That Have NO Match

In [0]:
emp_df.join(dept_df, "dept_id", "left_anti").show()

> Only Frank — the employee whose department doesn't exist. Opposite of left_semi. Useful for finding **orphan/unmatched** records.

#### 7. CROSS JOIN — Every Row Combined with Every Row

In [0]:
emp_df.crossJoin(dept_df).count()   # 6 employees × 4 depts = 24 rows
emp_df.crossJoin(dept_df).select("name", "dept_name").show(6)


> Extremely expensive on large data — use with caution.

#### 8. Join on Different Column Names

In [0]:
# When join columns have different names in each DataFrame
emp_df.join(dept_df, emp_df.dept_id == dept_df.dept_id, "inner") \
      .drop(dept_df.dept_id).show() # drop duplicate column

#### 9. Join on Multiple Columns

In [0]:
# Orders placed by customers
orders = [
    (1, "Alice",   "IN", "Electronics", 5000),
    (2, "Alice",   "US", "Clothing",    3000),  # same name, different country
    (3, "Bob",     "IN", "Clothing",    2000),
    (4, "Charlie", "EU", "Electronics", 8000),
    (5, "Diana",   "IN", "Electronics", 1000),  # no discount record
]
orders_df = spark.createDataFrame(orders, ["order_id", "name", "country", "category", "amount"])

# Discounts based on customer name AND country (both must match)
discounts = [
    ("Alice",   "IN", 10),   # Alice from IN gets 10% discount
    ("Alice",   "US", 15),   # Alice from US gets 15% discount
    ("Bob",     "IN",  5),   # Bob from IN gets 5% discount
    ("Charlie", "EU", 20),   # Charlie from EU gets 20% discount
    ("Eve",     "IN",  8),   # Eve has no orders
]
disc_df = spark.createDataFrame(discounts, ["name", "country", "discount_pct"])

##### 1. Join Using List (same column names in both DataFrames)

In [0]:
orders_df.join(disc_df, ["name", "country"], "inner").show()

3. Join Using Conditions (different column names)


##### 2. Left Join — Keep All Orders

In [0]:
orders_df.join(disc_df, ["name", "country"], "left").show()

##### 3. Join Using Conditions (different column names)
When column names differ between DataFrames, use explicit conditions:

In [0]:
# Rename columns to simulate different names
orders_df2  = orders_df.withColumnRenamed("name", "cust_name") \
                       .withColumnRenamed("country", "cust_country")

disc_df2    = disc_df.withColumnRenamed("name", "disc_name") \
                     .withColumnRenamed("country", "disc_country")

orders_df2.join(
    disc_df2,
    (orders_df2.cust_name    == disc_df2.disc_name) &
    (orders_df2.cust_country == disc_df2.disc_country),
    "inner"
).drop("disc_name", "disc_country").show()   # drop duplicate join cols

##### 4. Multi-Column Join with Additional Filter

In [0]:
from pyspark.sql.functions import col, round

orders_df.join(disc_df, ["name", "country"], "left") \
    .withColumn("disc_amount", round(col("amount") * col("discount_pct") / 100, 2)) \
    .withColumn("final_amount", col("amount") - col("disc_amount")) \
    .show()

##### 5. Three-Table Multi-Column Join

In [0]:
# Add a third table — shipping rates by country + category
shipping = [
    ("IN", "Electronics", 200),
    ("IN", "Clothing",    100),
    ("US", "Electronics", 300),
    ("EU", "Electronics", 250),
]
ship_df = spark.createDataFrame(shipping, ["country", "category", "ship_cost"])

orders_df \
    .join(disc_df,  ["name", "country"],        "left") \
    .join(ship_df,  ["country", "category"],    "left") \
    .show()

![image_1775382658979.png](./image_1775382658979.png "image_1775382658979.png")

#### 10. Broadcast Join — Performance Optimization

In [0]:
from pyspark.sql.functions import broadcast

# dept_df is small — broadcast it to all executors to avoid shuffle
emp_df.join(broadcast(dept_df), "dept_id", "inner").show()

- **Normal join**:   both sides shuffled across network (expensive)
- **Broadcast join**: small table sent to every executor, large table stays in place

> Use when one side is small (default threshold: 10 MB). Eliminates the shuffle entirely.

### Join Types — Visual Summary 

![image_1775383195654.png](./image_1775383195654.png "image_1775383195654.png")

### Unions

#### union() — By Position

In [0]:
df1 = spark.createDataFrame([("Alice", 30)], ["name", "age"])
df2 = spark.createDataFrame([("Bob",   25)], ["name", "age"])

df1.union(df2).show()

#### unionByName() — By Column Name (safer)

In [0]:
df1 = spark.createDataFrame([("Alice", 30)], ["Name", "Age"])
df2 = spark.createDataFrame([(25, "Bob")], ["Age", "Name"])

df1.unionByName(df2).show()

#### unionByName with Missing Columns

In [0]:
df1 = spark.createDataFrame([("Alice", 30)], ["Name", "Age"])
df2 = spark.createDataFrame([("Bob", "US")], ["Name", "Country"])

df1.unionByName(df2, allowMissingColumns = True).show()


**union()** matches columns by position, so the concept of "missing columns" doesn't apply — it has no awareness of column names at all. It simply expects both DataFrames to have the same number of columns.

**unionByName()** matches by name, so it can detect which columns are missing in each DataFrame and fill them with null.



> If DataFrames have different columns, always use unionByName(allowMissingColumns=True). Plain union() requires identical structure (same count, same order).

![image_1775384042412.png](./image_1775384042412.png "image_1775384042412.png")